In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import numpy as np
from rasterio.mask import mask
import rasterio
import os
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patheffects as path_effects

# ===============================
# Config
# ===============================

#files downloaded from https://eogdata.mines.edu/nighttime_light/annual/
RASTERS = {
    2014: "VNL_v21_npp_2014_global_vcmslcfg_c202205302300.average_masked.dat.tif",
    2015: "VNL_v21_npp_2015_global_vcmslcfg_c202205302300.average_masked.dat.tif",
    2016: "VNL_v21_npp_2016_global_vcmslcfg_c202205302300.average_masked.dat.tif",
    2017: "VNL_v21_npp_2017_global_vcmslcfg_c202205302300.average_masked.dat.tif",
    2018: "VNL_v21_npp_2018_global_vcmslcfg_c202205302300.average_masked.dat.tif",
    2019: "VNL_v21_npp_2019_global_vcmslcfg_c202205302300.average_masked.dat.tif",
    2020: "VNL_v21_npp_2020_global_vcmslcfg_c202205302300.average_masked.dat.tif",
    2021: "VNL_v21_npp_2021_global_vcmslcfg_c202205302300.average_masked.dat.tif",
    2022: "VNL_v22_npp-j01_2022_global_vcmslcfg_c202303062300.average_masked.dat.tif",
    2023: "VNL_npp_2023_global_vcmslcfg_v2_c202402081600.average_masked.dat.tif",
    2024: "VNL_npp_2024_global_vcmslcfg_v2_c202502261200.average_masked.dat.tif",
    2025:"VNL_npp_2025_global_vcmslcfg_v2_c202604011200.average_masked.dat.tif"
}

INDIA_PATH = "INDIA_STATES.geojson"
CITIES_PATH = "ne_10m_populated_places/ne_10m_populated_places.shp"
OUTPUT_BASE_DIR = "nightlights_states"

os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)

# ===============================
# Load India states
# ===============================
india_gdf = gpd.read_file(INDIA_PATH)

# ===============================
# Load cities
# ===============================
cities_gdf = gpd.read_file(CITIES_PATH)

# Ensure CRS match
if cities_gdf.crs != india_gdf.crs:
    cities_gdf = cities_gdf.to_crs(india_gdf.crs)

# Filter only India
for col in ["ADM0NAME", "SOV0NAME", "ADMIN", "ADM0_A3"]:
    if col in cities_gdf.columns:
        if col == "ADM0_A3":
            cities_gdf = cities_gdf[cities_gdf[col] == "IND"]
        else:
            cities_gdf = cities_gdf[cities_gdf[col] == "India"]
        break

# Find city name column
city_name_col = next((c for c in ["NAME", "NAMEASCII", "NAME_EN", "CITY"] if c in cities_gdf.columns), None)
if city_name_col is None:
    raise ValueError("No city name column found!")

# Select important cities
if "SCALERANK" in cities_gdf.columns:
    big_cities = cities_gdf[cities_gdf["SCALERANK"] <= 9]
elif "POP_MAX" in cities_gdf.columns:
    big_cities = cities_gdf.nlargest(50, "POP_MAX")
else:
    big_cities = cities_gdf.head(50)

# ===============================
# Tamil Nadu label tuning
# ===============================
CITY_OFFSETS = {
    "chennai": (0.01, 0.01),
    "coimbatore": (-0.02, 0.01),
    "madurai": (0.01, -0.02),
    "tiruchirappalli": (0.01, 0.01),
    "thanjavur": (0, -0.02),
}

# ===============================
# Colormap
# ===============================
colors_list = [
    (0, 0, 0.1), (0, 0, 0.3), (0.1, 0, 0.5),
    (0.4, 0.1, 0.6), (0.7, 0.2, 0.3),
    (1, 0.4, 0), (1, 0.8, 0),
    (1, 1, 0.8), (1, 1, 1)
]
cmap = LinearSegmentedColormap.from_list("nightlights", colors_list, N=256)

# ===============================
# Raster clip
# ===============================
def load_clip_raster(raster_path, geometry):
    with rasterio.open(raster_path) as src:
        raster, transform = mask(
            src, [geometry],
            crop=True,
            all_touched=True,
            nodata=0
        )
        meta = src.meta.copy()
        meta.update({
            "height": raster.shape[1],
            "width": raster.shape[2],
            "transform": transform,
            "nodata": 0
        })
    return raster[0], meta

# ===============================
# Plot
# ===============================
def plot_single(ax, nl, meta, year, geom, bounds, cities_subset):
    extent = [
        meta['transform'][2],
        meta['transform'][2] + meta['transform'][0] * meta['width'],
        meta['transform'][5] + meta['transform'][4] * meta['height'],
        meta['transform'][5]
    ]

    ax.imshow(
        nl,
        cmap=cmap,
        norm=colors.PowerNorm(
            gamma=0.35,
            vmin=1,
            vmax=np.nanpercentile(nl, 99)
        ),
        extent=extent
    )

    # Boundary
    gpd.GeoSeries([geom]).boundary.plot(
        ax=ax,
        edgecolor='white',
        linewidth=1.0
    )

    # Cities
    for _, city in cities_subset.iterrows():
        x, y = city.geometry.x, city.geometry.y
        name = str(city[city_name_col])

        ax.scatter(
            x, y,
            color="red",
            s=15,
            edgecolors='white',
            linewidth=0.4,
            zorder=10
        )

        name_l = name.lower()
        x_off = (extent[1] - extent[0]) * 0.01
        y_off = (extent[3] - extent[2]) * 0.01

        if name_l in CITY_OFFSETS:
            dx, dy = CITY_OFFSETS[name_l]
            x_off = (extent[1] - extent[0]) * dx
            y_off = (extent[3] - extent[2]) * dy

        ax.text(
            x + x_off, y + y_off,
            name,
            fontsize=8,
            color="white",
            weight='bold',
            path_effects=[
                path_effects.withStroke(linewidth=2, foreground="black")
            ]
        )

    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_facecolor("black")

    ax.text(
        0.12, 0.87,
        str(year),
        transform=ax.transAxes,
        fontsize=28,
        color='white',weight='bold',
        bbox=dict(facecolor='black', alpha=0.7)
    )

# ===============================
# Main
# ===============================
def plot_state_timeseries(state_name):

    # Robust state match
    state_row = None
    for col in ["STNAME_SH", "NAME", "STNAME", "STATE_NAME"]:
        if col in india_gdf.columns:
            row = india_gdf[
                india_gdf[col].str.strip().str.lower() == state_name.strip().lower()
            ]
            if not row.empty:
                state_row = row
                break

    if state_row is None:
        print("State not found")
        return

    geom = state_row.iloc[0].geometry
    bounds = geom.bounds

    # Filter cities inside state
    cities_in_state = big_cities[big_cities.intersects(geom)]

    # Output folder
    out_dir = os.path.join(
        OUTPUT_BASE_DIR,
        state_name.replace(" ", "_")
    )
    os.makedirs(out_dir, exist_ok=True)

    # Loop rasters
    for year, path in RASTERS.items():
        if not os.path.exists(path):
            print(f"Missing: {path}")
            continue

        nl, meta = load_clip_raster(path, geom)

        fig, ax = plt.subplots(figsize=(8, 10))
        plot_single(ax, nl, meta, year, geom, bounds, cities_in_state)

        fig.patch.set_facecolor("black")

        out_file = os.path.join(out_dir, f"{state_name}_{year}.png")
        plt.savefig(
            out_file,
            dpi=300,
            bbox_inches='tight',
            facecolor="black"
        )
        plt.close()

        print(f"Saved: {out_file}")

    print("Done!")

# ===============================
# Run
# ===============================
plot_state_timeseries("Tamil Nadu")

Saved: nightlights_states/Tamil_Nadu/Tamil Nadu_2014.png
Saved: nightlights_states/Tamil_Nadu/Tamil Nadu_2015.png
Saved: nightlights_states/Tamil_Nadu/Tamil Nadu_2016.png
Saved: nightlights_states/Tamil_Nadu/Tamil Nadu_2017.png
Saved: nightlights_states/Tamil_Nadu/Tamil Nadu_2018.png
Saved: nightlights_states/Tamil_Nadu/Tamil Nadu_2019.png
Saved: nightlights_states/Tamil_Nadu/Tamil Nadu_2020.png
Saved: nightlights_states/Tamil_Nadu/Tamil Nadu_2021.png
Saved: nightlights_states/Tamil_Nadu/Tamil Nadu_2022.png
Saved: nightlights_states/Tamil_Nadu/Tamil Nadu_2023.png
Saved: nightlights_states/Tamil_Nadu/Tamil Nadu_2024.png
Saved: nightlights_states/Tamil_Nadu/Tamil Nadu_2025.png
Done!
